<a href="https://colab.research.google.com/github/Andresdotdev/PZColab/blob/main/PZ_Colab_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🗺️ **Quick Notebook Guide**

| Cell | Action |
|---|---|
| 1 | 🚀 Install the server (choose b42 / b41 version) |
| 2 | 🔥 Encender servidor + reclamar túnel Playit + consola en vivo + apagado limpio (integrado) |
| 3 | 📦 Mods: inyección + descarga Workshop |
| 3.1 | 🩺 Diagnóstico de logs por mod |
| 4 | 💾 Backup de saves en Drive |

_Flujo típico: 1 → 2 (encender + jugar + apagar). Las celdas 3, 3.1 y 4 se usan bajo demanda._


In [ ]:
# @title 1. Install Server & Dependencies
# @markdown ---
# @markdown ### 🎮 Version Selection
Version = "b42 stable" # @param ["b42 stable", "b41 legacy", "b42 unstable"]

import os
import re
import json
import time
import subprocess
from IPython.display import clear_output

SAVES_PATH = "/content/drive/MyDrive/ZomboidSaves"
STATE_PATH = f"{SAVES_PATH}/.pzcolab_state.json"
SERVER_PATH = "/content/pzserver"

def beta_args(version):
    if version == "b41 legacy":
        return ["-beta", "legacy41"]
    if version == "b42 unstable":
        return ["-beta", "unstable"]
    return []

def mostrar_panel(etapa, progreso_steam=None):
    clear_output(wait=True)
    print("=========================================================")
    print("🚀 PROJECT ZOMBOID SERVER INSTALLER")
    print("=========================================================\n")
    pasos = [
        "1. Prepare system & dependencies",
        "2. Download Playit.gg tunnel agent",
        "3. Mount Google Drive",
        "4. Download Server (SteamCMD)",
    ]
    for i, p in enumerate(pasos, start=1):
        if etapa >= i:
            print(f"[✅] {p}")
        elif etapa == i - 1:
            print(f"[⏳] {p}")
        else:
            print(f"[  ] {p}")
    if progreso_steam and etapa == 3:
        print("\n   📊 DOWNLOAD PROGRESS:")
        for state, pct in progreso_steam.items():
            bar_length = 30
            filled = int(bar_length * pct / 100)
            bar = '█' * filled + '░' * (bar_length - filled)
            print(f"      ► {state:<14} |{bar}| {pct:>5.1f}%")

mostrar_panel(0)

# --- 1. SISTEMA ---
!sudo dpkg --add-architecture i386 > /dev/null 2>&1
!sudo apt update -y > /dev/null 2>&1
!echo steam steam/license note '' | debconf-set-selections
!echo steam steam/question select "I AGREE" | debconf-set-selections
!sudo apt install lib32gcc-s1 lib32stdc++6 steamcmd curl -y > /dev/null 2>&1
!/usr/games/steamcmd +quit > /dev/null 2>&1
mostrar_panel(1)

# --- 2. PLAYIT ---
!curl -sL https://github.com/playit-cloud/playit-agent/releases/download/v0.15.26/playit-linux-amd64 -o /usr/local/bin/playit > /dev/null 2>&1
!chmod +x /usr/local/bin/playit
mostrar_panel(2)

# --- 3. GOOGLE DRIVE ---
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(SAVES_PATH, exist_ok=True)
if os.path.exists("/root/Zomboid") and not os.path.islink("/root/Zomboid"):
    os.system("mv /root/Zomboid /root/Zomboid.local_$(date +%s)")
    print("⚠️ Old local /root/Zomboid folder renamed; replaced by the Drive link.")
os.system("rm -rf /root/Zomboid")
os.system(f'ln -s "{SAVES_PATH}" /root/Zomboid')
mostrar_panel(3, {"Iniciando...": 0.0})

# --- 4. STEAMCMD (idempotente: no re-descarga si ya está instalado) ---
ya_instalado = os.path.exists(f"{SERVER_PATH}/ProjectZomboid64")
misma_version = False
try:
    with open(STATE_PATH) as f:
        misma_version = json.load(f).get("version") == Version
except Exception:
    pass

if ya_instalado and misma_version:
    mostrar_panel(4)
    print("\nℹ️ The server is already installed with the selected version. Skipping download.")
else:
    if ya_instalado:
        print("⚠️ A different version is installed. Stopping the server and reinstalling...")
        os.system("pkill -f ProjectZomboid64 2>/dev/null")
        time.sleep(3)

    cmd_base = ['/usr/games/steamcmd', '+force_install_dir', SERVER_PATH, '+login', 'anonymous', '+app_update', '380870']
    cmd = cmd_base + beta_args(Version) + ['+quit']

    # Intento de descarga con retry: Colab free puede interrumpir por inactividad (~15 min sin output)
    # o rate-limits de Steam. Reintentamos limpiando pzserver.
    intento = 0
    max_intentos = 3
    ok_descarga = False
    while intento < max_intentos and not ok_descarga:
        intento += 1
        # Limpiar parciales del intento anterior
        os.system("rm -rf " + SERVER_PATH + " 2>/dev/null")
        os.makedirs(SERVER_PATH, exist_ok=True)
        if intento > 1:
            print(f"\n🔄 Download retry {intento}/{max_intentos} (cleaning previous state)...")
            time.sleep(5)

        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            progreso = {}
            try:
                for line in process.stdout:
                    line_lower = line.lower()
                    match = re.search(r'(?:stage|update state)\s+\(([^)]+)\)\s+([a-zA-Z]+),\s+progress:\s+([0-9.]+)', line_lower)
                    if match:
                        state = match.group(2).capitalize()
                        try:
                            percent = float(match.group(3))
                        except ValueError:
                            percent = 0.0
                        if percent > 100.0:
                            percent = 100.0
                        progreso[state] = percent
                        mostrar_panel(3, progreso)
            except KeyboardInterrupt:
                # Colab interrumpió: matar el proceso y reintentar
                print("\n⚠️ Download interrupted (possible Colab idle pause).")
                try:
                    process.terminate()
                    process.wait(timeout=30)
                except Exception:
                    pass
                continue
            rc = process.wait()
            if rc != 0:
                print(f"⚠️ SteamCMD exited with code {rc} (rate-limit or network).")
                continue
            ok_descarga = True
        except Exception as e:
            print(f"⚠️ SteamCMD error on attempt {intento}: {e}")
            continue

    if not ok_descarga:
        print(f"\n⚠️ Download failed after {max_intentos} attempts.")
        print("💡 Fix: Restart the Colab runtime (Disconnect and reconnect) and re-run this cell.")

# --- 4.1 VALIDAR INTEGRIDAD (si falta el ejecutable, validar vía SteamCMD) ---
if ok_descarga and not os.path.exists(f"{SERVER_PATH}/ProjectZomboid64"):
    print("\n🔎 Validating file integrity via SteamCMD (may take a couple of minutes)...")
    cmd_validate = ['/usr/games/steamcmd', '+force_install_dir', SERVER_PATH, '+login', 'anonymous', '+app_update', '380870', 'validate', '+quit']
    try:
        vproc = subprocess.run(cmd_validate, capture_output=True, text=True, timeout=600)
        mostrar_panel(3)
        if vproc.returncode != 0:
            print(f"⚠️ Validation failed (code {vproc.returncode}).")
    except Exception as e:
        print(f"⚠️ Could not validate integrity: {e}")

# --- 5. PARCHAR MEMORIA (Colab tiene ~12.7 GB de RAM) ---
START_SH = f"{SERVER_PATH}/start-server.sh"
if os.path.exists(START_SH):
    with open(START_SH) as f:
        contenido = f.read()
    nuevo = re.sub(r'-Xms\S+', '-Xms6g', contenido)
    nuevo = re.sub(r'-Xmx\S+', '-Xmx6g', nuevo)
    if nuevo != contenido:
        with open(START_SH, 'w') as f:
            f.write(nuevo)
        print("💾 Server memory adjusted to 6 GB (fits within Colab's limit).")
    else:
        print("ℹ️ start-server.sh already had memory configured (not modified).")
else:
    print("⚠️ start-server.sh was not found after installation.")

# --- 6. GUARDAR ESTADO (sincroniza la versión entre todas las celdas) ---
try:
    with open(STATE_PATH, 'w') as f:
        json.dump({"version": Version, "server_path": SERVER_PATH}, f)
    print("📌 State saved on Drive: all cells will use this version.")
except Exception as e:
    print(f"⚠️ Could not save state: {e}")

# --- FINALIZACIÓN ---
if os.path.exists(f"{SERVER_PATH}/ProjectZomboid64"):
    mostrar_panel(4)
    print("\n=========================================================")
    print("✅ PHASE 1 COMPLETED! You can continue to Cell 2.")
    print("=========================================================")
else:
    print("\n⚠️ SteamCMD failed to validate the executable. Try running the cell again.")


In [ ]:
# @title 2. Start Server + Playit Tunnel + Console (Auto-shutdown)
# @markdown ---
# @markdown ### 🎮 Server Parameters
server_name = 'PzColab' # @param {type: "string"}
admin_password = '' # @param {type: "string"}
server_password = '' # @param {type: "string"}
port = 16261 # @param {type: "integer"}
max_players = 16 # @param {type: "integer"}
pause_when_empty = True # @param {type: "boolean"}
# @markdown _💡 More memory fits more mods and players. Safe max: 8 GB on Colab._
memory_gb = "6 GB" # @param ["4 GB", "6 GB", "8 GB"]
# @markdown _💡 If you change the port, update the tunnel at playit.gg._
# @markdown
# @markdown ### 🛡️ Watchdog (auto-restart on crashes)
watchdog_enabled = True # @param {type: "boolean"}
max_restarts = 3 # @param {type: "integer"}

import os
import re
import json
import time
import sys
import secrets
import threading
import subprocess

SAVES_PATH = "/content/drive/MyDrive/ZomboidSaves"
SERVER_PATH = "/content/pzserver"
LOG_PATH = "/tmp/pzserver.log"
STATE_PATH = f"{SAVES_PATH}/.pzcolab_state.json"
INI_DIR = f"{SAVES_PATH}/Server"
INI_PATH = f"{INI_DIR}/{server_name}.ini"

def abortar(msg):
    print(msg)
    sys.exit()

# --- 0. VERIFICAR PRE-REQUISITOS ---
if not os.path.exists("/content/drive"):
    abortar("❌ Google Drive is NOT mounted. Run Cell 1 first.")

Version = "b42 estable"
if os.path.exists(STATE_PATH):
    try:
        with open(STATE_PATH) as f:
            Version = json.load(f).get("version", "b42 estable")
    except Exception:
        pass
print(f"📌 Active version: {Version}")

# --- 0.1 LIMPIAR EJECUCIONES ANTERIORES (re-ejecución segura de la celda) ---
if "pz_proc" in globals() and globals()["pz_proc"] and globals()["pz_proc"].poll() is None:
    if "parada" in globals() and globals()["parada"]:
        globals()["parada"].set()
    print("🛑 Previous server detected; stopping it before continuing.")
    os.system("pkill -f ProjectZomboid64 2>/dev/null")
    time.sleep(5)

# Recrear el symlink de saves si el runtime se reinició
if not os.path.islink("/root/Zomboid"):
    os.makedirs(SAVES_PATH, exist_ok=True)
    os.system("rm -rf /root/Zomboid")
    os.system(f'ln -s "{SAVES_PATH}" /root/Zomboid')
    print("🔗 Saves symlink recreated (new runtime).")

# --- 1. CONFIGURAR EL .INI (puerto, jugadores, pausa, passwords) ---
os.makedirs(INI_DIR, exist_ok=True)

def set_ini(key, valor):
    linea = f"{key}={valor}"
    if os.path.exists(INI_PATH):
        with open(INI_PATH) as f:
            lines = f.readlines()
        encontrado = False
        for i, l in enumerate(lines):
            if l.startswith(f"{key}="):
                lines[i] = linea + "\n"
                encontrado = True
                break
        if not encontrado:
            lines.append(linea + "\n")
        with open(INI_PATH, "w") as f:
            f.writelines(lines)
    else:
        with open(INI_PATH, "w") as f:
            f.write(linea + "\n")

if not admin_password:
    m = None
    if os.path.exists(INI_PATH):
        with open(INI_PATH) as f:
            m = re.search(r'^AdminPassword=(.*)', f.read(), re.MULTILINE)
    if m and m.group(1).strip():
        admin_password = m.group(1).strip()
        print("🔑 Admin password recovered from the existing .ini.")
    else:
        admin_password = "Pz" + secrets.token_hex(4)
        print(f"🔑 Admin password auto-generated: {admin_password}")

if not os.path.exists(INI_PATH):
    with open(INI_PATH, "w") as f:
        f.write(f"Port={port}\nDefaultPort={port}\nMaxPlayers={max_players}\nPauseOnEmpty={str(pause_when_empty).lower()}\nPassword={server_password}\nAdminPassword={admin_password}\n")
    print("ℹ️ First run: base .ini created with your settings.")
else:
    set_ini("Port", port)
    set_ini("DefaultPort", port)
    set_ini("MaxPlayers", max_players)
    set_ini("PauseOnEmpty", str(pause_when_empty).lower())
    set_ini("Password", server_password)
    set_ini("AdminPassword", admin_password)
    print(f"⚙️ Settings applied in {INI_PATH}")

if port != 16261:
    print(f"⚠️ Port changed to {port}. Update the tunnel at https://playit.gg/account")

# Guardar el nombre del servidor para que las demás celdas lo usen
try:
    with open(STATE_PATH) as f:
        estado = json.load(f)
    estado["server_name"] = server_name
    with open(STATE_PATH, "w") as f:
        json.dump(estado, f)
except Exception:
    pass

# --- 2. PLAYIT TÚNEL (reclamo inline primera vez / background después) ---
PLAYIT_DRIVE = f"{SAVES_PATH}/playitgg"
PLAYIT_CONFIG = "/root/.config/playit_gg"
playit_proc = None  # global del kernel, persiste entre re-ejecuciones
try:
    playit_proc = globals().get("playit_proc")
except Exception:
    pass

os.system("pkill -f 'playit' 2>/dev/null")

# Enlazar persistencia de config a Drive
if os.path.exists("/content/drive"):
    os.makedirs(PLAYIT_DRIVE, exist_ok=True)
    if os.path.isdir(PLAYIT_CONFIG) or os.path.islink(PLAYIT_CONFIG):
        os.system("rm -rf " + PLAYIT_CONFIG)
    os.system("ln -s " + PLAYIT_DRIVE + " " + PLAYIT_CONFIG)

config_existe = os.path.isdir(PLAYIT_DRIVE) and any(os.scandir(PLAYIT_DRIVE)) if os.path.isdir(PLAYIT_DRIVE) else False
if not config_existe:
    # Primera vez: reclamo foreground del túnel (la celda se pausa)
    print("🚀 First run: claim your Playit.gg tunnel in the window that opens.")
    print("⚠️ Authorize the link and return here. The cell will wait.")
    print("=" * 50)
    get_ipython().system("playit")
    config_existe = os.path.isdir(PLAYIT_DRIVE) and any(os.scandir(PLAYIT_DRIVE)) if os.path.isdir(PLAYIT_DRIVE) else False

if config_existe:
    playit_proc = subprocess.Popen(["playit"], stdout=open("/tmp/playit.log", "a"), stderr=subprocess.STDOUT)
    print("✅ Playit.gg tunnel running in the background (handle saved):", playit_proc.pid)
else:
    print("⚠️ Could not claim the Playit.gg tunnel. Server runs locally but is not reachable externally.")

# --- 3. APLICAR MEMORIA CONFIGURADA (parche Xms/Xmx en start-server.sh) ---
START_SH = "/content/pzserver/start-server.sh"
if not os.path.exists(START_SH):
    abortar("❌ start-server.sh not found. Run Cell 1 (installation) first.")

memoria_elegida = int(str(memory_gb).split()[0])
tope_seguro = 8
try:
    with open("/proc/meminfo") as f:
        for linea in f:
            if linea.startswith("MemTotal:"):
                ram_total_gb = int(linea.split()[1]) // 1024 // 1024
                tope_seguro = min(8, max(4, ram_total_gb - 4))
                break
except Exception:
    pass
memoria_final = min(memoria_elegida, tope_seguro)
if memoria_final < memoria_elegida:
    print(f"⚠️ You chose {memoria_elegida} GB but this runtime allows up to {tope_seguro} GB. Applying {memoria_final} GB.")

with open(START_SH) as f:
    contenido = f.read()
nuevo = re.sub(r'-Xms\S+', f'-Xms{memoria_final}g', contenido)
nuevo = re.sub(r'-Xmx\S+', f'-Xmx{memoria_final}g', nuevo)
if nuevo != contenido:
    with open(START_SH, "w") as f:
        f.write(nuevo)
    print(f"💾 Server memory: {memoria_final} GB (safe cap for this runtime: {tope_seguro} GB).")

os.system("chmod +x /content/pzserver/start-server.sh 2>/dev/null")

# --- 4. LANZAR SERVIDOR EN SEGUNDO PLANO CON WATCHDOG ---
logf = open(LOG_PATH, "a", buffering=1)
logf.write(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] 🚀 INICIO DE SERVIDOR ({Version})\n")
logf.flush()

parada = threading.Event()

def arrancar():
    return subprocess.Popen(
        ["/content/pzserver/start-server.sh", "-servername", server_name, "-adminpassword", admin_password],
        stdin=subprocess.PIPE, stdout=logf, stderr=subprocess.STDOUT, text=True)

pz_proc = arrancar()
reinicios = 0

def monitor():
    global pz_proc, reinicios
    while True:
        code = pz_proc.poll()
        if code is not None:
            logf.write(f"[{time.strftime('%H:%M:%S')}] ⚠️ El servidor terminó (código {code}).\n")
            logf.flush()
            if parada.is_set() or not watchdog_enabled or reinicios >= max_restarts:
                logf.write("[...] Watchdog detenido.\n")
                logf.flush()
                break
            reinicios += 1
            logf.write(f"[{time.strftime('%H:%M:%S')}] 🔄 Reinicio {reinicios}/{max_restarts} en 8s...\n")
            logf.flush()
            time.sleep(8)
            pz_proc = arrancar()
        time.sleep(5)

threading.Thread(target=monitor, daemon=True).start()

print("🔥 Server started in the background.")
print("📄 Live console integrated below (⏹ to stop and shut down cleanly).")
if watchdog_enabled:
    print(f"🛡️ Watchdog active: up to {max_restarts} automatic restarts.")

# --- 5. CONSOLA EN VIVO + APAGADO LIMPIO (flujo unificado) ---
# El tail se consume con subprocess para que el ⏹ de Colab dispare KeyboardInterrupt
# y entre en el bloque finally => save() -> quit() automático al terminar.
tail_proc = None
try:
    print(f"📖 Streaming de {LOG_PATH} — pulsa ⏹ para detener y apagar de forma limpia.")
    if os.path.exists(LOG_PATH):
        tail_proc = subprocess.Popen(["tail", "-n", "40", "-f", LOG_PATH],
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for linea in tail_proc.stdout:
            print(linea, end="", flush=True)
    else:
        print("⚠️ No log yet. Streaming will start once the server boots.")
        time.sleep(20)
except KeyboardInterrupt:
    print("\n🛑 Manual interruption detected: starting clean shutdown...")
except Exception as e:
    print(f"\n⚠️ Error en el tail: {e}")
finally:
    if tail_proc and tail_proc.poll() is None:
        tail_proc.terminate()
        try:
            tail_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tail_proc.kill()
    # Apagado ordenado del servidor: save -> quit
    if parada is not None:
        parada.set()
    if pz_proc and pz_proc.poll() is None and pz_proc.stdin:
        print("💾 Sending SAVE...")
        try:
            pz_proc.stdin.write("save\n")
            pz_proc.stdin.flush()
        except Exception:
            pass
        time.sleep(15)
        print("🛑 Sending QUIT...")
        try:
            pz_proc.stdin.write("quit\n")
            pz_proc.stdin.flush()
        except Exception:
            pass
        try:
            pz_proc.wait(timeout=90)
            print("✅ Server shut down safely.")
        except subprocess.TimeoutExpired:
            print("⚠️ No response in time; forcing shutdown.")
            pz_proc.terminate()
    else:
        os.system("pkill -TERM -f ProjectZomboid64 2>/dev/null; sleep 10; pkill -KILL -f ProjectZomboid64 2>/dev/null")
        print("✅ Termination signals sent.")
    if playit_proc and playit_proc.poll() is None:
        playit_proc.terminate()
        try:
            playit_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            playit_proc.kill()
    if logf and not logf.closed:
        logf.close()
    print("🎯 Complete shutdown: world saved and tunnel stopped.")


In [ ]:
# @title 4. Easy Mods: Paste URLs or Collections
# @markdown ---
# @markdown ### 🧹 History Control
clear_previous_list = False # @param {type:"boolean"}
# @markdown _💡 Enable it to clear old mods from the .ini and keep **only** the ones you paste below._
# @markdown
# @markdown ### 📥 Mod Input (one per line)
# @markdown _Paste the Workshop URL or just the numeric ID. If it's a collection, it expands automatically._
mods_input = "" # @param {type:"raw"}
# @markdown _Advanced format if auto-detection fails: `URL|ModIDManual`_
# @markdown
# @markdown ### 📥 Download Workshop Mods
download_mods = True # @param {type:"boolean"}
# @markdown _💡 Downloads each item via SteamCMD and detects the real Mod ID from its `mod.info`._
# @markdown
# @markdown **▶️ To confirm: run this cell with the ▶ button (or Ctrl+Enter). Form fields are processed when the cell runs — there is no internal button.**

import os, re, json, zipfile, subprocess

SAVES_PATH = '/content/drive/MyDrive/ZomboidSaves'
SERVER_PATH = '/content/pzserver'
STATE_PATH = f"{SAVES_PATH}/.pzcolab_state.json"
WS_APP = "108600"
WS_BASE = f"{SERVER_PATH}/steamapps/workshop/content/{WS_APP}"

try:
    with open(STATE_PATH) as f:
        estado = json.load(f)
    Version = estado.get("version")
    server_name = estado.get("server_name", "PzColab")
    if not Version:
        raise ValueError("sin version")
except Exception:
    Version = "b42 estable"
    server_name = "PzColab"
    print("⚠️ Cell 1 state not found (.pzcolab_state.json). Assuming b42 stable.\n")

is_b42 = Version.startswith("b42")
INI_PATH = f"{SAVES_PATH}/Server/{server_name}.ini"

print(f"📌 Version detected from Cell 1: {Version.upper()} | Server: {server_name}\n")

# --- 1. EXTRAER WORKSHOP ID DE CADA LÍNEA ---
def extraer_id(linea):
    linea = linea.strip()
    if not linea:
        return None
    manual = None
    if "|" in linea:
        linea, _, manual = linea.partition("|")
        linea = linea.strip()
        manual = manual.strip() or None
    m = re.search(r'id=(\d+)', linea) or re.search(r'(\d{5,})', linea)
    if not m:
        return None
    return (m.group(1), manual)

entradas = []
for l in mods_input.splitlines():
    r = extraer_id(l)
    if r:
        entradas.append(r)
    elif l.strip():
        print(f"⚠️ Line ignored (doesn't look like a Workshop ID): {l.strip()[:60]}")

if not entradas:
    print("ℹ️ No new mods to process. To see the current ones, use Cell 4.1.")
else:
    # --- 2. RESOLVER COLECCIONES + VERIFICAR COMPATIBILIDAD (1 request por item) ---
    try:
        import requests
    except ImportError:
        requests = None

    HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) PZColab/1.0"}

    def pagina_workshop(wsid):
        if requests is None:
            return None
        try:
            r = requests.get(f"https://steamcommunity.com/sharedfiles/filedetails/?id={wsid}", headers=HEADERS, timeout=20)
            if r.status_code == 200:
                return r.text
        except Exception:
            pass
        return None

    def es_coleccion(pagina):
        return bool(pagina) and "collectionChildren" in pagina

    def hijos_coleccion(wsid):
        if requests is None:
            return []
        for url in (f"https://steamcommunity.com/sharedfiles/filedetails/?id={wsid}&insideModal=1",
                    f"https://steamcommunity.com/sharedfiles/filedetails/?id={wsid}"):
            try:
                r = requests.get(url, headers=HEADERS, timeout=20)
                if r.status_code == 200:
                    m = re.search(r'<div[^>]*class="[^"]*collectionChildren[^"]*"[^>]*>(.*?)</div>', r.text, re.DOTALL)
                    bloque = m.group(1) if m else r.text
                    ids = re.findall(r'sharedfiles/filedetails/\?id=(\d+)', bloque)
                    if ids:
                        return list(dict.fromkeys(ids))
            except Exception:
                continue
        return []

    def analizar_compatibilidad(wsid, pagina):
        """Heurístico: avisa si la página del mod menciona una build distinta a la activa."""
        if not pagina:
            return None
        txt = pagina.lower()
        marca41 = bool(re.search(r'\bb41\b|build\s*41', txt))
        marca42 = bool(re.search(r'\bb42\b|build\s*42|42\.\d', txt))
        if marca41 and marca42:
            return None
        if marca42 and not is_b42:
            return f"the page mentions Build 42 but your server is {Version.upper()}"
        if marca41 and is_b42:
            return f"the page mentions Build 41 but your server is {Version.upper()}"
        return None

    final = []
    vistos = set()
    def agregar(wsid, manual):
        if wsid not in vistos:
            vistos.add(wsid)
            final.append((wsid, manual))

    avisos_compat = []
    cola = [(wsid, manual, 0) for wsid, manual in entradas]
    while cola:
        wsid, manual, prof = cola.pop(0)
        if manual:
            agregar(wsid, manual)
            continue
        pagina = pagina_workshop(wsid)
        if es_coleccion(pagina) and prof < 3:
            print(f"📂 Collection detected: {wsid} → expanding...")
            for h in hijos_coleccion(wsid):
                cola.append((h, None, prof + 1))
        else:
            agregar(wsid, None)
            aviso = analizar_compatibilidad(wsid, pagina)
            if aviso:
                avisos_compat.append((wsid, aviso))

    print(f"🧾 Items to process: {len(final)}\n")

    # --- 3. DESCARGAR ITEMS (opcional) ---
    if download_mods:
        if not os.path.exists(SERVER_PATH):
            print("⚠️ Server installation not found. Run Cell 1 to install.")
        else:
            print("📥 Downloading mods from Steam Workshop...")
            for wsid, manual in final:
                carpeta = f"{WS_BASE}/{wsid}"
                if os.path.isdir(carpeta) and list(os.scandir(carpeta)):
                    print(f"   ✓ {wsid} already downloaded (skipped).")
                    continue
                print(f"   → Workshop ID {wsid}...")
                cmd = ['/usr/games/steamcmd', '+force_install_dir', SERVER_PATH, '+login', 'anonymous',
                       '+workshop_download_item', WS_APP, wsid, '+quit']
                r = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                ok = r.returncode == 0 and os.path.isdir(carpeta) and list(os.scandir(carpeta))
                print(f"     {'✅ Downloaded' if ok else '⚠️ Failed (check the ID or use URL|ModIDManual)'}")
            print()

    # --- 4. DETECTAR MOD ID REAL DESDE mod.info ---
    def parse_mod_info(txt):
        mid = mname = None
        requires = []
        for line in txt.splitlines():
            line = line.strip()
            if line.startswith("id="):
                mid = line.partition("=")[2].strip()
            elif line.startswith("name="):
                mname = line.partition("=")[2].strip()
            elif line.startswith("require="):
                requires = [x.strip() for x in line.partition("=")[2].split(";") if x.strip()]
        return (mid, mname, requires)

    def detectar(wsid):
        base = f"{WS_BASE}/{wsid}"
        res = []
        if os.path.isdir(base):
            for root, dirs, files in os.walk(base):
                for fn in files:
                    if fn == "mod.info":
                        try:
                            with open(os.path.join(root, fn), encoding="utf-8", errors="ignore") as f:
                                res.append(parse_mod_info(f.read()))
                        except Exception:
                            pass
                    elif fn.lower().endswith(".zip"):
                        try:
                            with zipfile.ZipFile(os.path.join(root, fn)) as z:
                                for nombre in z.namelist():
                                    if nombre.endswith("mod.info"):
                                        res.append(parse_mod_info(z.read(nombre).decode("utf-8", errors="ignore")))
                        except Exception:
                            pass
        return [(mid, mname, requires) for mid, mname, requires in res if mid]

    def clasificar(nombre, mid):
        n = f"{nombre} {mid}".lower()
        if any(k in n for k in ("lib", "tsar", "core", "framework")): return "lib"
        if "ui" in n: return "ui"
        if any(k in n for k in ("car", "vehicle", "bike")): return "car"
        return "qol"

    nuevos = []  # (wsid, mod_id, tipo, nombre)
    requerimientos = {}
    for wsid, manual in final:
        detectados = detectar(wsid)
        if detectados:
            for mid, mname, requires in detectados:
                nuevos.append((wsid, mid, clasificar(mname, mid), mname or mid))
                if requires:
                    requerimientos[mid] = requires
        elif manual:
            nuevos.append((wsid, manual, clasificar(manual, manual), manual))
        else:
            print(f"⚠️ Could not detect the Mod ID for {wsid}. If you know it, use: URL|ModID")

    # --- 5. MERGE CON HISTORIAL DEL .ini (sin duplicados) ---
    if not os.path.exists(INI_PATH):
        print("❌ ERROR: INI file not found. Start the server once (Cell 3) to generate it.")
    else:
        with open(INI_PATH, 'r') as f:
            ini_lines = f.readlines()
        ini_content = "".join(ini_lines)

        base = []
        if not clear_previous_list:
            ws_match = re.search(r'^WorkshopItems=(.*)', ini_content, re.MULTILINE)
            mod_match = re.search(r'^Mods=(.*)', ini_content, re.MULTILINE)
            ws_list = [x.strip() for x in ws_match.group(1).split(';') if x.strip()] if ws_match and ws_match.group(1).strip() else []
            mod_list = [x.strip().replace('\\', '') for x in mod_match.group(1).split(';') if x.strip()] if mod_match and mod_match.group(1).strip() else []
            for i in range(min(len(ws_list), len(mod_list))):
                base.append((ws_list[i], mod_list[i], "qol", mod_list[i]))
        else:
            print("🧹 History cleared. Only pasted mods will be written.\n")

        combinada = list(nuevos)
        vistos_ws = {m[0] for m in combinada}
        for b in base:
            if b[0] not in vistos_ws:
                vistos_ws.add(b[0])
                combinada.append(b)

        # Orden de carga: librerías primero, luego UI, vehículos y QoL
        peso = {"lib": 0, "ui": 1, "car": 2, "qol": 3}
        combinada.sort(key=lambda m: peso.get(m[2], 3))

        if not combinada:
            print("⚠️ No active mods to write to the server.")
        else:
            ws_ids = [m[0] for m in combinada]
            mod_ids = [m[1] for m in combinada]
            workshop_str = f"WorkshopItems={';'.join(ws_ids)}\n"
            mod_str = f"Mods={';'.join([f'\\{m}' if is_b42 else m for m in mod_ids])}\n"

            ws_found, mod_found = False, False
            for idx, line in enumerate(ini_lines):
                if line.startswith("WorkshopItems="):
                    ini_lines[idx] = workshop_str
                    ws_found = True
                elif line.startswith("Mods="):
                    ini_lines[idx] = mod_str
                    mod_found = True
            if not ws_found:
                ini_lines.append(workshop_str)
            if not mod_found:
                ini_lines.append(mod_str)

            with open(INI_PATH, 'w') as f:
                f.writelines(ini_lines)

            # --- 6. REPORTE CON NOMBRES REALES ---
            iconos = {"lib": "📚", "ui": "🖥️", "car": "🚗", "qol": "⚙️"}
            totales = {}
            for m in combinada:
                totales[m[2]] = totales.get(m[2], 0) + 1
            print("=" * 60)
            print(f"📋 MODS ON THE SERVER (Total: {len(combinada)})")
            print(f"📊 Resumen: {totales.get('lib', 0)} lib · {totales.get('ui', 0)} ui · {totales.get('car', 0)} car · {totales.get('qol', 0)} qol")
            print("=" * 60)
            for wsid, mid, tipo, nombre in combinada:
                print(f"   {iconos.get(tipo, '⚙️')} {nombre} ({tipo}) | Workshop: {wsid}")
                reqs = requerimientos.get(mid)
                if reqs:
                    print(f"      🔗 Requires: {', '.join(reqs)}")
            print("-" * 60)

            faltantes = []
            ids_configurados = {m[1] for m in combinada}
            for mid, reqs in requerimientos.items():
                for req in reqs:
                    if req not in ids_configurados:
                        faltantes.append((mid, req))
            if faltantes:
                print("\n⚠️ MISSING DEPENDENCIES:")
                for mid, req in faltantes:
                    print(f"   Mod '{mid}' requires '{req}', which is not in the list. Add it or the server may fail to load.")

            if avisos_compat:
                print("\n🔎 POSSIBLE VERSION INCOMPATIBILITIES (heuristic, verify on the Workshop):")
                for wsid, motivo in avisos_compat:
                    print(f"   ⚠️ Workshop {wsid}: {motivo}")
                print("   If the mod fails to load, check its page to confirm compatibility.")

            print(f"✅ .ini updated: {INI_PATH}")
            print("   Restart the server (Cell 3) to apply the mods.")


In [ ]:
# @title 🔍 4.1 Server Inspector & Advanced Diagnostics
# @markdown _Shows your active mods and analyzes logs, grouping errors by mod to find the real culprits._

import os, re, json

SAVES_PATH = '/content/drive/MyDrive/ZomboidSaves'
try:
    with open(f"{SAVES_PATH}/.pzcolab_state.json") as f:
        server_name = json.load(f).get("server_name", "PzColab")
except Exception:
    server_name = "PzColab"
INI_PATH = f"{SAVES_PATH}/Server/{server_name}.ini"

# --- PARTE 1: LECTURA DE MODS ---
print("=========================================================")
print("👁️  MODS CONFIGURED ON THE SERVER")
print("=========================================================")

if not os.path.exists(INI_PATH):
    print("❌ No configuration .ini file found.")
else:
    with open(INI_PATH, 'r') as f:
        content = f.read()

    ws_match = re.search(r'^WorkshopItems=(.*)', content, re.MULTILINE)
    mod_match = re.search(r'^Mods=(.*)', content, re.MULTILINE)

    ws_list = [x.strip() for x in ws_match.group(1).split(';') if x.strip()] if ws_match and ws_match.group(1).strip() else []
    mod_list = [x.strip().replace('\\', '') for x in mod_match.group(1).split(';') if x.strip()] if mod_match and mod_match.group(1).strip() else []
    total_mods = min(len(ws_list), len(mod_list))

    def nombre_amigable(wsid, fallback):
        base = f"/content/pzserver/steamapps/workshop/content/108600/{wsid}"
        if os.path.isdir(base):
            for root, dirs, files in os.walk(base):
                if "mod.info" in files:
                    try:
                        with open(os.path.join(root, "mod.info"), encoding="utf-8", errors="ignore") as f:
                            for line in f:
                                if line.strip().startswith("name="):
                                    return line.partition("=")[2].strip() or fallback
                    except Exception:
                        pass
        return fallback

    if total_mods > 0:
        for i in range(total_mods):
            mid = mod_list[i]
            tag = "📦 Mod"
            if "lib" in mid.lower() or "tsar" in mid.lower(): tag = "📚 Lib"
            elif "ui" in mid.lower(): tag = "🖥️ UI"
            elif "car" in mid.lower() or "vehicle" in mid.lower(): tag = "🚗 Car"
            print(f"[{i+1:>2}] {tag} ➡️ {nombre_amigable(ws_list[i], mid):<28} | Workshop: {ws_list[i]}")
    else:
        print("⚠️ The .ini file has no mods configured.")
print("=========================================================\n")

# --- PARTE 2: DIAGNÓSTICO INTELIGENTE ---
print("🔍 STARTING ADVANCED LOG SCAN...")
print("=========================================================")

log_files = []
for subdir in ("Server", "Logs"):
    base = os.path.join(SAVES_PATH, subdir)
    if not os.path.isdir(base):
        continue
    for root, dirs, files in os.walk(base):
        for file in files:
            if file.endswith('.txt') and ('debuglog' in file.lower() or 'console' in file.lower()):
                log_files.append(os.path.join(root, file))

if not log_files:
    print("ℹ️ No active log files found.")
else:
    ULTIMO_LOG = max(log_files, key=os.path.getmtime)
    print(f"📖 Analizando incidencias en: MIdrive/{ULTIMO_LOG.replace('/content/drive/MyDrive/', '')}\n")

    with open(ULTIMO_LOG, 'r', encoding='utf-8', errors='ignore') as f:
        log_lines = f.readlines()

    fallos_criticos = []
    alertas_esteticas = []
    errores_steam = []
    problemas_memoria = []
    errores_servidor = []
    fallos_guardado = []

    # Diccionario para contar qué mods están dando más guerra
    mods_culpables = {}

    for idx, line in enumerate(log_lines):
        line_lower = line.lower()
        num_linea = idx + 1

        # 1. DETECTAR CRASHES O ERRORES LUA
        if "lua error" in line_lower or "stack trace" in line_lower or "call stack" in line_lower:
            # Rastrear contexto (buscar el nombre del mod culpable en las 4 líneas cercanas)
            contexto = "Desconocido (Script interno)"
            for k in range(max(0, idx-2), min(len(log_lines), idx+3)):
                match_mod = re.search(r'(media/lua/[^\s]+|mods/([^/\s]+))', log_lines[k])
                if match_mod:
                    contexto = match_mod.group(1)
                    break

            fallos_criticos.append((num_linea, line.strip(), contexto))
            mods_culpables[contexto] = mods_culpables.get(contexto, 0) + 1

        # 2. DETECTAR FALLOS DE STEAM WORKSHOP
        elif "workshop" in line_lower and ("fail" in line_lower or "error" in line_lower or "rejected" in line_lower):
            errores_steam.append(f"[Line {num_linea}] 🌐 Steam Failure ➡️ {line.strip()}")

        # 3. DETECTAR PROBLEMAS DE MEMORIA
        elif ("outofmemory" in line_lower or "out of memory" in line_lower or "gc overhead" in line_lower
              or "heap space" in line_lower or "could not reserve enough space" in line_lower
              or "insufficient memory" in line_lower):
            problemas_memoria.append(f"[Line {num_linea}] 🧠 ➡️ {line.strip()[:110]}")

        # 4. DETECTAR ERRORES GENERALES DEL SERVIDOR (puertos, assert, steam)
        elif ("failed to bind" in line_lower or "address already in use" in line_lower
              or "assertion failed" in line_lower or "illegal worker thread" in line_lower
              or ("steam" in line_lower and ("timeout" in line_lower or "not responding" in line_lower))):
            errores_servidor.append(f"[Line {num_linea}] ⚠️ ➡️ {line.strip()[:110]}")

        # 5. DETECTAR FALLOS AL GUARDAR
        elif "failed to save" in line_lower or ("save" in line_lower and ("corrupt" in line_lower or "failed" in line_lower)):
            fallos_guardado.append(f"[Line {num_linea}] 💾 ➡️ {line.strip()[:110]}")

        # 6. FILTRAR ALERTAS MENORES (Evita alarmar por sonidos o vallas rotas)
        elif "missing" in line_lower and ("thumpsound" in line_lower or "tile" in line_lower or "media/sound" in line_lower):
            alertas_esteticas.append(f"[Line {num_linea}] 📝 Detail ➡️ {line.strip()[:90]}...")

    # --- DESPLIEGUE DEL REPORTE RESUMIDO ---
    if fallos_criticos or errores_steam or alertas_esteticas or problemas_memoria or errores_servidor or fallos_guardado:

        if problemas_memoria:
            print(f"🧠 MEMORY PROBLEMS: {len(problemas_memoria)}")
            for err in problemas_memoria[:3]: print(f"   {err}")
            print("   💡 Increase the memory in Cell 3 (max 8 GB) or reduce MaxPlayers/heavy mods.")
            print("-" * 60)

        if errores_servidor:
            print(f"⚠️ SERVER ERRORS: {len(errores_servidor)}")
            for err in errores_servidor[:3]: print(f"   {err}")
            print("-" * 60)

        if fallos_guardado:
            print(f"💾 SAVE FAILURES: {len(fallos_guardado)}")
            for err in fallos_guardado[:3]: print(f"   {err}")
            print("-" * 60)

        if fallos_criticos:
            print(f"🔴 Lua Errors/Crashes Detected: {len(fallos_criticos)}")
            print("👑 MOST UNSTABLE MODS OR FILES:")
            for mod, count in sorted(mods_culpables.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"   ⚠️ -> '{mod}' triggered {count} alerts on this boot.")
            print("\n📌 Sample of the first error lines:")
            for num, _, ctx in fallos_criticos[:4]:
                print(f"   [Line {num}] Script: {ctx}")
            print("-" * 60)

        if errores_steam:
            print(f"\n🌐 Steam Workshop Issues: {len(errores_steam)}")
            for err in errores_steam[:3]: print(f"   {err}")
            print("-" * 60)

        if alertas_esteticas:
            print(f"\n📝 Minor/Aesthetic Alerts (don't break the server): {len(alertas_esteticas)}")
            print("   💡 _Note: Missing sounds or original map fences. Ignorable._")
            for err in alertas_esteticas[:3]: print(f"   {err}")
            print("-" * 60)

        print("\n🛠️ DIAGNÓSTICO FINAL:")
        if fallos_criticos:
            print("   The server started, but some mods have outdated scripts. If you notice visual lag or invisible items,")
            print("   check the mods listed at the top of the instability ranking.")
        elif problemas_memoria:
            print("   The server ran out of memory. Increase the memory in Cell 3 (max 8 GB) or reduce players/mods.")
        elif errores_servidor:
            print("   There are network/port or server errors. Review the lines flagged above.")
        elif fallos_guardado:
            print("   There were errors saving the world. Check Drive space and use Cell 3.2 for a clean shutdown.")
        else:
            print("   Stable! No critical mod scripting issues recorded.")
    else:
        print("✅ 100% CLEAN! Spotless logs, ready to play.")

print("=========================================================")


In [ ]:
# @title 5. Saves Backup (Drive)
# @markdown _Creates a .tar.gz backup of the world and config on your Google Drive._
backup_max_keep = 3 # @param {type: "integer"}

import os, glob, time, tarfile

SAVES_PATH = "/content/drive/MyDrive/ZomboidSaves"
BACKUP_DIR = "/content/drive/MyDrive/ZomboidSaves_backups"

if not os.path.exists("/content/drive"):
    print("❌ Drive not mounted. Run Cell 1 first.")
else:
    os.makedirs(BACKUP_DIR, exist_ok=True)
    ts = time.strftime("%Y%m%d_%H%M%S")
    destino = f"{BACKUP_DIR}/ZomboidSaves_{ts}.tar.gz"
    print("📦 Creating backup (may take a few minutes depending on world size)...")
    with tarfile.open(destino, "w:gz") as tar:
        tar.add(SAVES_PATH, arcname="ZomboidSaves", recursive=True)
    tamaño_mb = os.path.getsize(destino) / (1024 * 1024)
    print(f"✅ Backup created: {destino} ({tamaño_mb:.1f} MB)")

    backups = sorted(glob.glob(f"{BACKUP_DIR}/ZomboidSaves_*.tar.gz"))
    max_guardar = max(1, backup_max_keep)
    for viejo in backups[:-max_guardar]:
        os.remove(viejo)
    print(f"📊 Backups kept: {len(backups)} (max {max_guardar})")
    print(f"📂 Backup folder: {BACKUP_DIR}")


🛠️ Browser Anti-Idle Script
This script simulates clicking on the Colab page automatically every 10 minutes to fool the idle-detection system.

Steps to activate it:

1.   Open your Google Colab notebook in the browser (Chrome, Edge or Firefox).
2.   Press the F12 key (or right-click anywhere on the page and select Inspect).
3.   Go to the Console tab.
4.   Paste the following code and press Enter:
```
function KeepAlive() {
    console.log("Keeping server alive...");
    // Simulates a click on the connect or system-options button
    let connectButton = document.querySelector("#connect") || document.querySelector("colab-connect-button");
    if (connectButton) {
        connectButton.click();
    }
}
setInterval(KeepAlive, 600000); // Runs automatically every 10 minutes (600,000 ms)
```
You'll see a message in the console every 10 minutes. As long as you keep that browser tab open (even minimized), the server won't drop due to inactivity.
